# Notebook 05: Datasets - Loading, Transforming, and Visualizing Data

---

## What This Notebook Covers

This notebook builds the **data layer** that every later notebook quietly relies on: the code that turns a folder of raw images on the Hugging Face Hub into clean, batched `(x, y)` tensors a model can train on, plus the little plotting toolkit we use to *look* at that data. We will learn:

1. **Loading data with Hugging Face Datasets** - inspecting metadata *before* downloading, then loading Fashion-MNIST
2. **Indexing & batch slicing** - the difference between `ds[0]` (one sample) and `ds[:5]` (a batch), and why slicing "transposes" the data
3. **Collate functions** - how a list of samples becomes a stacked batch tensor
4. **Lazy transforms** - applying `PIL → Tensor` conversions on-access with `.with_transform()`
5. **Two reusable Python patterns** - the `@inplace` decorator and the `collate_dict` closure
6. **A plotting toolkit** - `show_image`, `subplots`, `get_grid`, `show_images` for displaying images in tidy grids
7. **A `DataLoaders` wrapper** - bundling train + validation loaders behind one object

By the end, a single line `DataLoaders.from_dd(dsd, batch_size=128)` will give us everything the training loop needs.

---

## Why a Whole Notebook on *Data*?

It is tempting to think the "real" deep learning is the model and the optimizer. In practice, **the data pipeline is where most bugs and most wasted time live.** A model that is silently fed images in the wrong shape, labels that are off-by-one, or pixels that were never scaled from `0–255` to `0–1` will train to garbage and give you no error message. So we invest here, once, and build tools we trust.

There are three recurring jobs this notebook solves cleanly:

- **Get the data** without writing bespoke download/caching code (Hugging Face Datasets handles it).
- **Shape the data** - PIL images must become float tensors of the right dtype, range, and axis order before a network will accept them.
- **See the data** - the single most effective debugging habit in deep learning is to *plot a batch* and check it looks right. We build a one-liner for that.

> **Climate / EO bridge.** This is exactly the workflow you already run on ECOSTRESS or ERA5 tiles: a loader that reads rasters lazily, a transform that normalizes bands and reorders axes to channels-first, and a `show_images`-style grid to eyeball whether the patches and masks line up before you trust a model's loss. The library here is `datasets` instead of `rioxarray`, but the shape of the problem - lazy read → transform → batch → visualize - is identical.

---

## Prerequisites

You should be comfortable with:

- **Python**: functions, classes, `*args`/`**kwargs`, list comprehensions, and a first taste of **decorators** and **closures** (we give full deep-dives on these, so a rough idea is enough).
- **PyTorch tensors**: shapes, `dtype`, and the idea of a `DataLoader` that yields batches (we built a minimal one by hand in notebook 04).
- **PIL images**: that an image has a width, height, and channel count, and that `0` is black and `255` is white for an 8-bit grayscale image.

This notebook exports several functions to the `miniai.datasets` module (every cell tagged `#|export`), so the tools we build here become importable in every notebook that follows.

---

# Part 1: Setup and Imports

---

We start by importing everything and configuring PyTorch/matplotlib for clean output. There are two import cells on purpose: the **first is exported** to `miniai.datasets` (it holds only what the *library* needs), and the **second is notebook-only** (extra tools we use for exploration but don't want compiled into the module).

In [ ]:
#|default_exp datasets

**What does the code above do?**

`#|default_exp datasets` is an **nbdev directive**. It tells nbdev "when you export this notebook, put all the `#|export` cells into a module file called `datasets.py`." Combined with the package name, that becomes `miniai/datasets.py`. It is a comment to Python (so it does nothing when you just run the cell) but a command to the nbdev build tool. Leave it untouched - changing it would send our exported code to the wrong file.

In [ ]:
#|export
from __future__ import annotations
import math,numpy as np,matplotlib.pyplot as plt
from operator import itemgetter
from itertools import zip_longest
import fastcore.all as fc

from torch.utils.data import default_collate

from miniai.training import *

**What does the code above do?**

This is the **exported** import block - it becomes the top of `miniai/datasets.py`, so it lists only what the library's functions actually need.

| Import | Why we need it |
|---|---|
| `from __future__ import annotations` | Lets us write type hints like `def f() -> DataLoaders:` *inside* the class that's still being defined. It makes all annotations lazy strings, so the names don't have to exist yet. |
| `math`, `numpy as np` | `math.sqrt` and `np.floor`/`np.ceil` for the grid-sizing math in `get_grid`. |
| `matplotlib.pyplot as plt` | All the plotting (`show_image`, `subplots`). |
| `itemgetter` (from `operator`) | A fast, C-implemented way to pull several keys out of a dict at once - the engine inside `collate_dict`. We deep-dive this later. |
| `zip_longest` (from `itertools`) | Like `zip`, but pads the shorter iterable with `None` instead of stopping early - used to pair images with titles even when there are fewer titles than images. |
| `fastcore.all as fc` | fast.ai's utility belt. We use `fc.delegates` (signature inheritance) and `fc.hasattrs`. |
| `default_collate` | PyTorch's built-in "stack a list of samples into a batch" function - we wrap it rather than reinvent it. |
| `from miniai.training import *` | Pulls in `get_dls` and other helpers we built in notebook 04. |

**Why `import *` here?** This is a deliberate exception to the usual "never `import *`" rule. `miniai` is *our own* library built notebook-by-notebook, and each notebook is meant to layer on top of the previous one's exports. Pulling in everything keeps the teaching code uncluttered.

In [ ]:
# Notebook-only imports (NOT exported to the module) - tools for exploring,
# timing, and testing that the library itself doesn't need.
import logging,pickle,gzip,os,time,shutil,torch,matplotlib as mpl
from pathlib import Path

from torch import tensor,nn,optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
from datasets import load_dataset,load_dataset_builder      # Hugging Face

import torchvision.transforms.functional as TF              # TF.to_tensor: PIL -> tensor
from fastcore.test import test_close

**What does the code above do?**

These imports are **not** marked `#|export`, so they stay in the notebook and never reach `miniai/datasets.py`. They are the things we need to *play with* the data here, not to *ship*.

The two that matter most for this notebook:

- **`load_dataset` / `load_dataset_builder`** from Hugging Face `datasets`. `load_dataset_builder` fetches only the *metadata* (description, features, split sizes) - no image bytes. `load_dataset` does the real download (and caches it, so it's instant next time).
- **`torchvision.transforms.functional as TF`**. The one function we use, `TF.to_tensor`, does two jobs at once: it converts a PIL image to a tensor **and** rescales pixels from integers `0–255` to floats `0.0–1.0`, while reordering axes from `H×W×C` to `C×H×W`. We visualize this exact conversion below.

In [ ]:
# Make tensors and plots readable, and make randomness reproducible.
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)  # 2 dp, wide, no 1e-3
torch.manual_seed(1)                  # same seed -> same "random" numbers every run
mpl.rcParams['image.cmap'] = 'gray'   # default colormap for imshow: black->white

**What does the code above do?**

Three quality-of-life settings:

| Setting | Effect |
|---|---|
| `set_printoptions(precision=2, linewidth=140, sci_mode=False)` | When you print a tensor, show 2 decimals, wrap at 140 chars, and write `0.001` rather than `1e-3`. Pure readability. |
| `manual_seed(1)` | Fixes PyTorch's random number generator so shuffling, weight init, etc. are identical on every run - essential when you want to compare two experiments and know the *only* difference was the change you made. |
| `mpl.rcParams['image.cmap'] = 'gray'` | Sets the default matplotlib colormap to grayscale, which is what we want for single-channel Fashion-MNIST. Without it, matplotlib would apply its default `viridis` and our clothes would come out purple-green. |

In [ ]:
# Hugging Face is chatty (download/cache/processing messages). Silence anything
# at WARNING level or below so the notebook output stays focused on results.
logging.disable(logging.WARNING)

**What does the code above do?**

The Hugging Face libraries emit a stream of `INFO`/`WARNING` log lines about downloading, caching, and processing. `logging.disable(logging.WARNING)` suppresses every message at `WARNING` severity *or lower* (`DEBUG`, `INFO`, `WARNING`), leaving only `ERROR` and `CRITICAL`. It keeps our output clean. In production you'd usually keep warnings on - they sometimes tell you something is genuinely wrong.

---

# Part 2: Loading Data with Hugging Face Datasets

---

**Hugging Face Datasets** is a library that gives you thousands of ready-made datasets behind one consistent API. It handles the annoying parts for you:

- **Downloading** from the Hugging Face Hub
- **Caching** locally, so you download once and load instantly forever after
- **Memory efficiency** - data is memory-mapped via Apache Arrow, so a 60,000-image dataset doesn't have to fit in RAM all at once
- **Lazy preprocessing** - transforms run on access, not up front (we exploit this heavily later)

**Why Fashion-MNIST?** It's a drop-in replacement for the classic MNIST digits: the same `28×28` grayscale format and the same `60,000 train / 10,000 test` split, but the task (classify 10 kinds of clothing) is harder than reading digits, which makes it a more honest benchmark. Same plumbing, more interesting problem.

Two key functions:

- `load_dataset_builder(name)` - fetch **metadata only** (no image download). Great for peeking before you commit.
- `load_dataset(name)` - download and load the actual data.

In [ ]:
# load_dataset_builder fetches METADATA ONLY - no images are downloaded here.
# Useful for checking what's inside a dataset before committing to the download.
name = "fashion_mnist"                  # any Hub id works: "mnist", "cifar10", ...
ds_builder = load_dataset_builder(name)
print(ds_builder.info.description)      # human-readable description of the dataset

**What does the code above do?**

`load_dataset_builder("fashion_mnist")` returns a **builder** object that knows everything *about* the dataset without having downloaded a single image. `ds_builder.info` is the metadata bundle; `.description` is the prose summary the dataset authors wrote. This is the cheap, polite way to explore: you can read the description, check the size, and inspect the label names before deciding to pull gigabytes to disk.

In [ ]:
# .features describes the SCHEMA: what fields each sample has and their types.
ds_builder.info.features

**What does the code above do?**

`.features` is the dataset's **schema** - the shape of one sample. For Fashion-MNIST it reads roughly:

```python
{'image': Image(decode=True, ...),
 'label': ClassLabel(num_classes=10,
                     names=['T-shirt/top','Trouser','Pullover','Dress','Coat',
                            'Sandal','Shirt','Sneaker','Bag','Ankle boot'])}
```

Two fields:

- **`image`** - type `Image`. `decode=True` means when you read a sample you get a ready-to-use **PIL image**, not raw PNG bytes.
- **`label`** - type `ClassLabel`. This is richer than a plain integer: it stores `num_classes=10` *and* a `names` list that maps each integer to a human-readable class. So label `9` "knows" it means `'Ankle boot'`. We use that mapping later to put real titles on our plots.

In [ ]:
# .splits describes how the data is partitioned and how big each part is.
ds_builder.info.splits

**What does the code above do?**

`.splits` reports the partitions: a `train` split with `60,000` examples and a `test` split with `10,000`. The golden rule: **train on `train`, report final numbers on `test`, and never let the model learn from `test`.** Fashion-MNIST ships no `validation` split, so in real training we'd carve one out of `train` (or just use `test` as a stand-in validation set for a teaching example, as we effectively do).

In [ ]:
# load_dataset actually downloads + caches the data, returning a DatasetDict
# (a dict-like keyed by split name). First run downloads; later runs hit cache.
dsd = load_dataset(name)
dsd

**What does the code above do?**

`load_dataset(name)` does the real work and returns a **`DatasetDict`** - think "a dictionary of splits." Its keys are split names (`'train'`, `'test'`) and its values are `Dataset` objects holding the actual samples. The first call downloads from the Hub and caches to disk; every call after that loads from cache and is effectively instant. Printing `dsd` shows the structure: each split, its `features`, and its number of rows.

In [ ]:
# A DatasetDict behaves like a normal dict; pull out the two splits.
train, test = dsd['train'], dsd['test']
train[0]      # one sample = a dict: {'image': <PIL image>, 'label': 9}

**What does the code above do?**

`dsd['train']` and `dsd['test']` give us the two `Dataset` objects. Each `Dataset` indexes like a list: `train[0]` is the first sample, `train[-1]` the last, `train[0:5]` the first five.

Crucially, **one sample is a dictionary**: `train[0]` returns `{'image': <PIL 28×28 grayscale>, 'label': 9}`. The keys are exactly the feature names we saw in the schema. Label `9` is an `'Ankle boot'`.

### 🎮 Interactive: the lifecycle of a dataset — metadata first, data later

Loading a dataset isn't one step, it's a little assembly line — and a key idea is that **nothing heavy is downloaded until you actually ask for the data**. In the widget below, **click any stage chip** to jump straight to it and reveal the exact line of code that produces it, or press **▶ Play** to watch the whole pipeline build itself (use the **speed slider** to slow it down). Watch the order: *builder* (metadata only) → *features* (the schema) → *splits* (sizes) → *load* (the real download + cache) → *unpack* (grab the two splits).

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. Loads ./interactive_viz/dataset_lifecycle.html in an
# isolated <iframe> (its CSS/JS can't leak into the notebook). The widget
# broadcasts its own height, so it shows FULL-WIDTH with NO inner scrollbar.
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# To use the published copy instead, swap src for:
#   https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/dataset_lifecycle.html
# ============================================================================
from IPython.display import HTML
HTML('''
<iframe id="frame-dataset_lifecycle"
        src="interactive_viz/dataset_lifecycle.html"
        style="width:100%; min-height:320px; border:1px solid #dde5f2; border-radius:14px;
               box-shadow:0 8px 24px rgba(123,92,214,.12); background:#eef3fb;"
        loading="lazy" title="Dataset lifecycle: builder, features, splits, load, unpack"></iframe>
<script>
addEventListener("message", function(e){
  if (e.data && e.data.type === "viz-height" && e.data.id === "dataset_lifecycle") {
    var f = document.getElementById("frame-dataset_lifecycle");
    if (f) f.style.height = (e.data.height + 2) + "px";
  }
});
</script>
''')

## Indexing by feature name (instead of hard-coding strings)

A small but tidy habit: rather than sprinkling the literal strings `'image'` and `'label'` throughout the code, we grab the feature names *once* into variables `x` and `y`. If a dataset used different field names tomorrow, we'd change them in one place.

In [ ]:
# The features object is dict-like; unpacking it yields its KEYS (the names).
# Tuple unpacking: first key -> x, second key -> y.
x, y = ds_builder.info.features   # x='image', y='label'

**What does the code above do?**

When you iterate over (or unpack) a dict, Python hands you its **keys**. Since `features` is dict-like with keys `'image'` and `'label'`, the line `x, y = ds_builder.info.features` assigns `x = 'image'` and `y = 'label'`. It's the same mechanism as `a, b = {'p':1, 'q':2}` giving `a='p', b='q'`. We now have the field names in variables.

In [ ]:
x, y       # confirm: ('image', 'label')

**What does the code above do?**

A quick sanity check - it should print `('image', 'label')`. Confirming assumptions immediately, with a one-line cell, is cheaper than debugging a wrong assumption ten cells later.

In [ ]:
# Hard-set the names (defensive: doesn't depend on dict order) and grab the
# first image. Jupyter renders a PIL image inline, so we just see the picture.
x, y = 'image', 'label'
img = train[0][x]    # train[0] -> dict; [x] -> the PIL image
img

**What does the code above do?**

We pin `x, y = 'image', 'label'` explicitly (so nothing depends on dictionary ordering), then read the first image: `train[0]` is the sample dict, and `[x]` (i.e. `['image']`) pulls the PIL image out of it. Because Jupyter knows how to display PIL images, evaluating `img` on the last line **renders the actual picture** - a tiny `28×28` grayscale boot. The PIL object also reports `mode='L'` (luminance = grayscale) and `size=28×28`.

## Single sample vs. a batch: `ds[0]` vs `ds[:5]`

Here is a distinction that trips up everyone once. Indexing with a single integer gives one sample (a dict). Indexing with a **slice** gives a *batch* - but the batch is **transposed** into a dict of *lists*.

In [ ]:
# Slicing returns a BATCH, transposed into a dict-of-lists.
#   train[0]   -> {'image': img,        'label': 9}            (one dict)
#   train[:5]  -> {'image': [img0..4],  'label': [9,0,0,3,0]}  (dict of lists)
xb = train[:5][x]   # list of 5 PIL images
yb = train[:5][y]   # list of 5 integer labels
yb

**What does the code above do?**

This is the key idea. Compare:

| Expression | Result | Shape of result |
|---|---|---|
| `train[0]` | `{'image': img, 'label': 9}` | one dict (a single sample) |
| `train[:5]` | `{'image': [img0,...,img4], 'label': [9,0,0,3,0]}` | a dict whose **values are lists** |

When you slice, Hugging Face does not give you a *list of dicts* `[{...}, {...}, ...]`; it gives you a **dict of lists**. The data has been "transposed": instead of grouping by sample, it groups by field. This is more efficient for batch work - all the images are already together in one list, all the labels in another. So `train[:5][x]` is "the list of the first five images" and `train[:5][y]` is "the list of the first five labels," which prints as `[9, 0, 0, 3, 0]`.

### 🎮 Interactive: slicing a dataset into a batch

Watch how `ds[:5]` pulls a contiguous run of samples out of the dataset and groups them into a batch. The order is just the index range - no shuffling here, that happens later in the DataLoader.

In [ ]:
from IPython.display import IFrame
IFrame("https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/minibatch_slicing.html", width="100%", height=520)

In [ ]:
# The ClassLabel feature carries the integer->name mapping.
featy = train.features[y]   # ClassLabel(num_classes=10, names=[...])
featy

**What does the code above do?**

`train.features` is the live schema of the *loaded* dataset, and `train.features['label']` is the **`ClassLabel`** object for the label field. It bundles `num_classes=10` with the `names` list (`'T-shirt/top'`, ..., `'Ankle boot'`). It's the dictionary that translates the model's numeric world back into words we can read.

In [ ]:
# int2str turns label indices into class names (single int or a whole list).
featy.int2str(yb)    # [9,0,0,3,0] -> ['Ankle boot','T-shirt/top',...]

**What does the code above do?**

`featy.int2str(yb)` maps each integer label to its class name. Given `yb = [9, 0, 0, 3, 0]` it returns `['Ankle boot', 'T-shirt/top', 'T-shirt/top', 'Dress', 'T-shirt/top']`. This is what lets us label a plot with *'Ankle boot'* instead of an inscrutable *9* - vital when you're eyeballing predictions and need to know what the model actually said.

### 🎮 Interactive: `.features`, `ClassLabel`, and the `int2str` decoder

Labels are stored as bare integers (cheap and model-ready), while the **`ClassLabel` feature** quietly remembers the human-readable names. **Click a class on the left**, or **drag the dial on the right**, and watch `int2str` translate an index like <span style='color:#f6921e;font-weight:700'>9</span> into <span style='color:#16a3a3;font-weight:700'>"Ankle boot"</span>. This is exactly the trick that lets us put readable titles on plots instead of inscrutable numbers.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. Loads ./interactive_viz/schema_features.html in an
# isolated <iframe> (its CSS/JS can't leak into the notebook). The widget
# broadcasts its own height, so it shows FULL-WIDTH with NO inner scrollbar.
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# To use the published copy instead, swap src for:
#   https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/schema_features.html
# ============================================================================
from IPython.display import HTML
HTML('''
<iframe id="frame-schema_features"
        src="interactive_viz/schema_features.html"
        style="width:100%; min-height:520px; border:1px solid #dde5f2; border-radius:14px;
               box-shadow:0 8px 24px rgba(123,92,214,.12); background:#eef3fb;"
        loading="lazy" title="Features schema and the int2str label decoder"></iframe>
<script>
addEventListener("message", function(e){
  if (e.data && e.data.type === "viz-height" && e.data.id === "schema_features") {
    var f = document.getElementById("frame-schema_features");
    if (f) f.style.height = (e.data.height + 8) + "px";
  }
});
</script>
''')

In [ ]:
# Two equivalent access orders (row-first vs column-first):
#   train[:5]['label']   slice rows, then take a column
#   train['label'][:5]   take the whole column, then slice  (used here)
train['label'][:5]

**What does the code above do?**

There are two ways to reach the same five labels:

- **Row-first**: `train[:5]['label']` - select the first five samples, *then* take their `label` field.
- **Column-first**: `train['label'][:5]` - take the entire `label` column (all 60,000), *then* slice the first five.

Both yield `[9, 0, 0, 3, 0]`. Column-first is handy when you want *all* values of one field (e.g. to compute class balance); row-first is better when you want specific samples. Hugging Face's Arrow backing makes even `train['label']` (all 60k) cheap, because it isn't copying images - just reading one column.

---

# Part 3: From Samples to Batches - the Collate Function

---

A PyTorch `DataLoader`'s job is to hand your training loop **batches**. But the dataset gives out individual samples, and our samples are *PIL images inside dicts* - which PyTorch can't stack into a tensor by itself. The bridge is a **collate function**: you give the `DataLoader` a function that takes a list of samples and returns one batched object.

We'll write a custom one, see its limitation, then discover a cleaner approach with transforms.

In [ ]:
# A collate function takes a LIST of samples and returns ONE batch.
# Default collate can't handle PIL images or our dict layout, so we write our own:
#   - convert each PIL image to a tensor and STACK into [batch, C, H, W]
#   - gather the integer labels into a 1-D tensor [batch]
def collate_fn(b):
    return {
        x: torch.stack([TF.to_tensor(o[x]) for o in b]),  # images -> [B,1,28,28]
        y: tensor([o[y] for o in b])                       # labels -> [B]
    }

**What does the code above do?**

`collate_fn(b)` receives `b`, a **list of sample dicts** - e.g. `[{'image': PIL, 'label': 9}, {'image': PIL, 'label': 0}, ...]` - and returns a single batched dict.

Reading the two lines:

- `torch.stack([TF.to_tensor(o[x]) for o in b])`: for each sample `o`, pull its PIL image `o[x]`, convert it to a tensor with `TF.to_tensor` (which also scales `0–255 → 0.0–1.0` and reorders to `C×H×W`), giving a list of `[1, 28, 28]` tensors. `torch.stack` then piles them into one `[B, 1, 28, 28]` tensor (a new batch axis at the front).
- `tensor([o[y] for o in b])`: collect the integer labels into a Python list and wrap it as a 1-D tensor of shape `[B]`.

The output is `{'image': <[B,1,28,28] tensor>, 'label': <[B] tensor>}` - exactly what a network wants.

In [ ]:
# Feed our collate_fn to a DataLoader and pull the first batch of 16.
dl = DataLoader(train, collate_fn=collate_fn, batch_size=16)
b = next(iter(dl))           # iter(dl) -> iterator; next(...) -> first batch
b[x].shape, b[y]             # [16,1,28,28], and the 16 labels

**What does the code above do?**

We hand `collate_fn` to a `DataLoader` with `batch_size=16`. `iter(dl)` creates an iterator over batches; `next(...)` pulls the first one. The result confirms the pipeline works:

- `b[x].shape` is `torch.Size([16, 1, 28, 28])` - 16 images, 1 channel, 28×28 pixels.
- `b[y]` is a length-16 tensor of labels.

Our PIL-dict samples have become clean batched tensors.

### 🎮 Interactive: building a custom `collate_fn`, one step at a time

The DataLoader hands your collate function a **plain list of samples** — and the function's job is to fuse them into one batch. **Step through the 5 stages** (or press **▶ Play**, with **speed control**): each clickable stage shows the matching line of code and an animation of what's happening — `TF.to_tensor` converting each PIL image, `torch.stack` piling them into `[B,1,28,28]`, the labels gathered into `[B]`, and finally the assembled batch dict.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. Loads ./interactive_viz/collate_fn_builder.html in an
# isolated <iframe> (its CSS/JS can't leak into the notebook). The widget
# broadcasts its own height, so it shows FULL-WIDTH with NO inner scrollbar.
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# To use the published copy instead, swap src for:
#   https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/collate_fn_builder.html
# ============================================================================
from IPython.display import HTML
HTML('''
<iframe id="frame-collate_fn_builder"
        src="interactive_viz/collate_fn_builder.html"
        style="width:100%; min-height:520px; border:1px solid #dde5f2; border-radius:14px;
               box-shadow:0 8px 24px rgba(123,92,214,.12); background:#eef3fb;"
        loading="lazy" title="Inside a custom collate_fn: list of samples to one batch"></iframe>
<script>
addEventListener("message", function(e){
  if (e.data && e.data.type === "viz-height" && e.data.id === "collate_fn_builder") {
    var f = document.getElementById("frame-collate_fn_builder");
    if (f) f.style.height = (e.data.height + 8) + "px";
  }
});
</script>
''')

### 🎮 Interactive: the DataLoader assembly line

The whole pipeline in one place: a **Sampler** chooses the order of indices, a **BatchSampler** chunks them into groups of `batch_size`, and the **collate** function fetches those samples and stacks them into the `(images, labels)` batch your loop consumes. Click a batch to watch it gather.

In [ ]:
from IPython.display import IFrame
IFrame("https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/dataloader_pipeline.html", width="100%", height=800)

---

# Part 4: A Cleaner Approach - Dataset Transforms

---

Doing the PIL→tensor conversion *inside* `collate_fn` works, but it tangles two responsibilities: "how to convert a sample" and "how to batch samples." Hugging Face offers a cleaner split with **`.with_transform(fn)`**: you attach a transform to the dataset itself, and it runs **lazily** - only when a sample is actually read. Then the *default* collate can do the batching, because by the time it sees the data, the images are already tensors.

In [ ]:
# A transform receives a BATCH dict and returns it modified.
# Here: replace the list of PIL images with a list of tensors.
def transforms(b):
    b[x] = [TF.to_tensor(o) for o in b[x]]   # PIL -> tensor, in place
    return b

**What does the code above do?**

`transforms(b)` takes a batch dict `b` (Hugging Face always passes transforms *batched* data, even when you access a single item) and rewrites its `image` field: every PIL image `o` in `b[x]` becomes a tensor via `TF.to_tensor`. It mutates `b` and then **returns** it - note that return, because the next section is all about what happens when you forget it.

In [ ]:
# with_transform attaches the transform LAZILY: original 'train' is untouched;
# 'tds' applies `transforms` on access. Now default collate handles batching.
tds = train.with_transform(transforms)
dl = DataLoader(tds, batch_size=16)   # no custom collate_fn needed!
b = next(iter(dl))
b[x].shape, b[y]                       # same result: [16,1,28,28], [16]

**What does the code above do?**

`train.with_transform(transforms)` returns a **new view**, `tds`, that applies `transforms` whenever a sample is accessed. The original `train` is unchanged and still yields PIL images. Two payoffs:

1. **No custom collate needed.** Because `tds` already produces tensors, PyTorch's *default* collate can stack them. `DataLoader(tds, batch_size=16)` just works.
2. **Clean separation.** "How to convert a sample" now lives in the transform; "how to batch" stays with the DataLoader. Each piece is simpler and reusable.

The batch shape is identical to before (`[16, 1, 28, 28]`), but the code is tidier.

### 🎮 Interactive: to_tensor - from PIL image to a model-ready tensor

Step through the exact conversion `torch.flatten(TF.to_tensor(img))`: the integer `0–255` PIL grid is rescaled to floats in `[0, 1]`, reordered from height-width-channel to **channel-first** `[1, 28, 28]`, and (for a linear model) unrolled into a flat `[784]` vector. Watch the shape annotation change at each stage.

In [ ]:
from IPython.display import IFrame
IFrame("https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/pil_to_tensor_pipeline.html", width="100%", height=560)

### 🎮 Interactive: with_transform is lazy - nothing happens until you look

Click individual samples to 'access' them. Only the samples you touch get converted from PIL to tensor; the rest stay in their compact original form. This on-demand behavior is why attaching a transform to a 60,000-image dataset is instant - the work is deferred until a DataLoader actually pulls a sample.

In [ ]:
from IPython.display import IFrame
IFrame("https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/with_transform_lazy.html", width="100%", height=560)

## Flattening, and a subtle bug: the missing `return`

For a plain fully-connected network (no convolutions), each image must be a flat `[784]` vector, not a `[1, 28, 28]` grid (`28×28 = 784`). We add `torch.flatten` to the transform. But this is where a classic Python footgun appears.

In [ ]:
# This transform FLATTENS each image to [784]... but note: NO return statement!
# It modifies b in place and (accidentally) returns None. That breaks with_transform.
def _transformi(b):
    b[x] = [torch.flatten(TF.to_tensor(o)) for o in b[x]]

**What does the code above do?**

`_transformi(b)` does the right computation - convert each PIL image to a tensor and `torch.flatten` it from `[1, 28, 28]` to `[784]` - **but it has no `return` statement.** In Python, a function that falls off the end without returning gives back `None`. And `.with_transform` *uses the returned value* as the transformed sample. So this transform would hand `None` to the pipeline and break it, even though the line that modifies `b` is perfectly correct. The leading underscore in `_transformi` is a hint that this is a "draft" we're about to fix.

The fix is a tiny, reusable decorator. It's worth a proper deep-dive, because **decorators** are one of those Python features that feel magical until you've seen one built from scratch.

### 🎮 Interactive: an image tensor in 3D — why flattening is just *unrolling*

Before we flatten an image to `[784]`, it helps to *see* what flattening does. This **3D view** (powered by three.js — **drag to rotate, scroll to zoom**) draws each pixel of a `[1,28,28]` image as a height bar. **Click the stages** (or press ▶ Play) to watch the 28×28 grid **unroll row-by-row into one flat line of 784 numbers**, then fold back. The punchline for the bug we're about to meet: flattening changes the *shape*, never the *values* — and the order is fully reversible with `.view(28,28)`.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. Loads ./interactive_viz/tensor_to_image_3d.html in an
# isolated <iframe> (its CSS/JS can't leak into the notebook). The widget
# broadcasts its own height, so it shows FULL-WIDTH with NO inner scrollbar.
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# To use the published copy instead, swap src for:
#   https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/tensor_to_image_3d.html
# ============================================================================
from IPython.display import HTML
HTML('''
<iframe id="frame-tensor_to_image_3d"
        src="interactive_viz/tensor_to_image_3d.html"
        style="width:100%; min-height:520px; border:1px solid #dde5f2; border-radius:14px;
               box-shadow:0 8px 24px rgba(123,92,214,.12); background:#eef3fb;"
        loading="lazy" title="3D image tensor: grid to flat 784 and back"></iframe>
<script>
addEventListener("message", function(e){
  if (e.data && e.data.type === "viz-height" && e.data.id === "tensor_to_image_3d") {
    var f = document.getElementById("frame-tensor_to_image_3d");
    if (f) f.style.height = (e.data.height + 8) + "px";
  }
});
</script>
''')

---

# Deep Dive: the `@inplace` Decorator

---

## The Problem `inplace` Solves

We keep wanting to write transform functions that **modify a batch in place** - `b[x] = something`. That's natural and readable. But Hugging Face's `.with_transform` doesn't look at the side effect; it looks at the **return value**. So every such function has to end with `return b`, and forgetting that one line produces a silent `None` bug with no error message.

Here is the painful "before":

```python
def transform_a(b):
    b['image'] = [...]
    return b            # easy to forget

def transform_b(b):
    b['image'] = [...]
    return b            # ...have to remember it every single time

def transform_c(b):
    b['image'] = [...]
    # oops - forgot the return. Silent bug.
```

What we *want* is to write the interesting line and have the boring `return b` added automatically:

```python
@inplace
def transformi(b):
    b['image'] = [...]   # no return needed - the decorator handles it
```

That is exactly what a **decorator** lets us do.

---

## What Is a Decorator? (the one-paragraph version)

A decorator is just **a function that takes a function and returns a new function**. In Python, `@inplace` written above `def transformi` is pure syntactic sugar for:

```python
transformi = inplace(transformi)
```

That's the whole secret. `inplace` receives your function, wraps it in a slightly enhanced version, and you get the enhanced version back under the same name. Nothing more mystical than "a function factory for functions."

---

## The Complete Code

```python
#|export
def inplace(f):
    def _f(b):
        f(b)        # run the original function (it mutates b in place)
        return b    # then hand b back - the part everyone forgets
    return _f       # return the wrapper to stand in for f
```

Four lines. Let's take them one at a time.

---

### Line 1: `def inplace(f):`

```python
def inplace(f):
```

**What it does:** Defines `inplace` as a function whose single argument `f` is *another function* - the transform you want to enhance (e.g. `_transformi`).

**Why this design:** Functions in Python are ordinary values. You can pass them as arguments, store them in variables, and return them, exactly like ints or strings. That "functions are values" fact is what makes decorators possible at all.

---

### Line 2: `def _f(b):` - defining the wrapper

```python
    def _f(b):
```

**What it does:** *Inside* `inplace`, we define a brand-new function `_f` that takes the batch `b`. This inner function is the "enhanced" version we'll hand back.

**Why this design:** `_f` is a **closure** - it's defined inside `inplace`, so it can see and remember `inplace`'s local variable `f` even after `inplace` has finished running. Each time you call `inplace(some_func)`, you get a fresh `_f` that has captured that specific `some_func` as its personal `f`. (We deep-dive closures again in `collate_dict` below - they're a recurring trick.)

---

### Line 3: `f(b)` then `return b` - the actual enhancement

```python
        f(b)        # call the original; it mutates b in place
        return b    # return the mutated b
```

**What it does:** When someone calls the wrapper `_f(b)`, it first runs your original function `f(b)` (which does the in-place modification, e.g. converting the images), and then it does the one thing your function "forgot": **`return b`**.

**Why this design:** This is the entire value of the decorator - it guarantees the `return b` is always there, so you never have to write it. Your transform can be a single expressive line, and the wrapper supplies the plumbing.

A subtle but important point: `f(b)` modifies the *same* dict object `b` that gets returned. Python passes objects by reference, so the mutation `f` performs is visible in the `b` we return. We don't need `f`'s return value at all (it's `None`); we only care that `f` changed `b`, and then we return `b` ourselves.

---

### Line 4: `return _f` - hand back the wrapper

```python
    return _f
```

**What it does:** `inplace` returns the wrapper function `_f` (note: `_f`, **not** `_f(b)` - we return the *function itself*, uncalled).

**Why this design:** The whole point is to replace your function with the enhanced one. After `transformi = inplace(transformi)`, the name `transformi` now points at `_f`. Calling `transformi(b)` runs the wrapper, which runs your original logic and then returns `b`.

---

## Putting It Together

```python
transformi = inplace(_transformi)
# Now transformi(b) does:  _transformi(b)  then  return b
```

Three things happen on that single line:
1. `inplace` is called with `_transformi` as its `f`.
2. It builds a fresh wrapper `_f` that remembers `f = _transformi`.
3. It returns `_f`, which we store under the name `transformi`.

From now on, `transformi` behaves like `_transformi` **plus** an automatic `return b`. The silent-`None` bug is impossible.

The interactive below shows the two paths side by side - the same image-converting line, but one returns `None` and the other (thanks to the decorator) returns `b`.

### 🎮 Interactive: why @inplace? a missing return silently breaks the pipeline

Press **Run transform** and watch both paths. *Both* functions correctly convert the images in `b` - the only difference is the **return value**. Without `@inplace`, the function returns `None` and the dataset breaks; with `@inplace`, the wrapper adds `return b` for you and the batch flows through.

In [ ]:
from IPython.display import IFrame
IFrame("https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/inplace_decorator.html", width="100%", height=560)

In [ ]:
#|export
def inplace(f):
    def _f(b):
        f(b)       # run the original (mutates b in place)
        return b   # supply the return the user 'forgot'
    return _f      # the wrapper stands in for f

**What does the code above do?**

This is the exported, production copy of the decorator we just dissected. It is tagged `#|export`, so it lands in `miniai.datasets` and becomes reusable everywhere. `inplace(f)` returns a wrapper that calls `f(b)` for its side effect, then returns `b`.

In [ ]:
# Apply it manually (no @ syntax yet): wrap the buggy _transformi.
transformi = inplace(_transformi)

**What does the code above do?**

`transformi = inplace(_transformi)` is the **manual** form of decoration: we call `inplace` ourselves and store the wrapped result. After this line, `transformi` is the fixed version - same flattening logic, but with the guaranteed `return b`. The `@inplace` syntax (coming up) does exactly this, just more prettily.

In [ ]:
# Verify the fix: accessing a sample now returns a real, flattened tensor.
r = train.with_transform(transformi)[0]   # apply transform, read sample 0
r[x].shape, r[y]                            # [784] (flattened!) and label 9

**What does the code above do?**

We attach the now-fixed `transformi` and read sample `0`. The result `r` is a proper dict again (not `None`), and `r[x].shape` is `torch.Size([784])` - the `28×28` image flattened to a 784-vector - with `r[y] == 9`. The decorator did its job: the missing-`return` bug is gone.

In [ ]:
# The clean, idiomatic form: @inplace does `transformi = inplace(transformi)` for us.
@inplace
def transformi(b):
    b[x] = [torch.flatten(TF.to_tensor(o)) for o in b[x]]

**What does the code above do?**

This is the **decorator syntax** - the form you'll actually use. Writing `@inplace` directly above `def transformi` tells Python: "after defining `transformi`, immediately replace it with `inplace(transformi)`." It is identical in effect to the manual `transformi = inplace(_transformi)` we wrote above, but it reads top-to-bottom, keeps the wrapping visible at the point of definition, and is the standard Python idiom. From here on, every transform we write gets `@inplace` and skips the boilerplate `return b`.

In [ ]:
# Confirm the @-syntax version behaves identically.
tdsf = train.with_transform(transformi)   # 'f' for flattened
r = tdsf[0]
r[x].shape, r[y]                           # [784] and label 9

**What does the code above do?**

A final check that the decorator-syntax `transformi` works the same as the manual one: build the flattened dataset `tdsf`, read sample 0, and confirm `[784]` and label `9`. We now have a clean, flattened, lazily-transformed dataset and a reusable `@inplace` tool in the library.

---

# Part 5: From Dict Batches to Tuple Batches

---

Our batches are still **dicts** (`{'image': ..., 'label': ...}`), but training loops are nicest with **tuples**, so you can write `for xb, yb in dl:`. To convert cleanly, we'll use a small standard-library tool, `itemgetter`, wrapped in a closure called `collate_dict`. First, a deep-dive on `itemgetter` and the duck-typing idea behind it.

---

# Deep Dive: `itemgetter` and Duck Typing

---

## The Problem `itemgetter` Solves

Constantly we need to pull a *few specific keys* out of a dict, as a tuple. The obvious way is a lambda:

```python
get = lambda d: (d['image'], d['label'])
```

That works, but it's verbose, it's slower than it needs to be, and it hard-codes the keys. `operator.itemgetter` does the same thing more cleanly and faster:

```python
from operator import itemgetter
get = itemgetter('image', 'label')   # a reusable "getter" object
get(d)                                # -> (d['image'], d['label'])
```

---

## How It Works

`itemgetter('a', 'c')` builds a small **callable object**. When you later call it on something, it returns a tuple of that object indexed at each key: `(obj['a'], obj['c'])`. It's implemented in C inside CPython, so it's faster than an equivalent Python lambda, and it reads as exactly what it is: "get these items."

```python
d = {'a': 1, 'b': 2, 'c': 3}
itemgetter('a', 'c')(d)    # -> (1, 3)
```

---

## The Subtle Superpower: Duck Typing

Here's the part that makes `itemgetter` perfect for our use. It does **not** require a dict. It requires only that the object support the `[]` operator - i.e. that it implements the special method `__getitem__`. Anything that does "quacks like a dict" as far as `itemgetter` is concerned.

This is **duck typing**: *"If it walks like a duck and quacks like a duck, treat it as a duck."* Python doesn't check the object's type; it just tries `obj[key]` and trusts it to work.

We can prove it with a tiny custom class that implements `__getitem__` but is in no way a dict:

```python
class D:
    def __getitem__(self, k):
        return 1 if k=='a' else 2 if k=='b' else 3

itemgetter('a', 'c')(D())   # -> (1, 3), exactly as with a real dict
```

`itemgetter` called `D()['a']` and `D()['c']`, which Python routed to `D.__getitem__`, which returned `1` and `3`.

**Why this matters for us:** a Hugging Face batch behaves like a dict (it supports `batch['image']`), and `default_collate`'s output behaves like a dict too. So we can point `itemgetter(*ds.features)` at the collated batch and it will extract the fields in feature order - no matter the exact concrete type - giving us the tuple our training loop wants. That is the engine of `collate_dict`, next.

In [ ]:
# itemgetter builds a reusable getter: ig(d) -> (d['a'], d['c'])
d = dict(a=1, b=2, c=3)
ig = itemgetter('a', 'c')
ig(d)        # -> (1, 3)

**What does the code above do?**

We build `ig = itemgetter('a', 'c')` once, then apply it: `ig(d)` returns `(d['a'], d['c']) == (1, 3)`. The getter is reusable - define it once, call it on as many dicts as you like.

In [ ]:
# Duck typing: itemgetter only needs [] access (__getitem__), not a real dict.
class D:
    def __getitem__(self, k):
        return 1 if k == 'a' else 2 if k == 'b' else 3

**What does the code above do?**

`D` is a deliberately weird class: it stores nothing, but it implements `__getitem__`, so `D()['a']` returns `1`, `D()['b']` returns `2`, and anything else returns `3`. It "quacks like a dict" for indexing purposes, despite being nothing of the sort.

In [ ]:
# The SAME ig works on D() - it just calls D().__getitem__ under the hood.
d = D()
ig(d)        # -> (1, 3)

**What does the code above do?**

We reuse the *same* `ig = itemgetter('a', 'c')` on an instance of `D`. It returns `(1, 3)`: `ig` called `d['a']` (→ `1`) and `d['c']` (→ `3`, since `'c'` is neither `'a'` nor `'b'`). The point is proven - `itemgetter` cares only about `[]` access, which is exactly why it works on Hugging Face batches and collated outputs alike.

### 🎮 Interactive: `itemgetter` + duck typing — one getter, any object

`itemgetter('a','c')` bakes your chosen keys into a tiny reusable function that returns a **tuple** in that order. **Click the keys** to change what it grabs, then **flip between a real `dict` and a custom `D()` object**: the *same* getter works on both, because Python only cares that the object supports `[]` access (`__getitem__`), not what *type* it is. That's *duck typing*, and it's the reason this one helper can power `collate_dict` on Hugging Face batches.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. Loads ./interactive_viz/itemgetter_ducktyping.html in an
# isolated <iframe> (its CSS/JS can't leak into the notebook). The widget
# broadcasts its own height, so it shows FULL-WIDTH with NO inner scrollbar.
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# To use the published copy instead, swap src for:
#   https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/itemgetter_ducktyping.html
# ============================================================================
from IPython.display import HTML
HTML('''
<iframe id="frame-itemgetter_ducktyping"
        src="interactive_viz/itemgetter_ducktyping.html"
        style="width:100%; min-height:520px; border:1px solid #dde5f2; border-radius:14px;
               box-shadow:0 8px 24px rgba(123,92,214,.12); background:#eef3fb;"
        loading="lazy" title="itemgetter and duck typing"></iframe>
<script>
addEventListener("message", function(e){
  if (e.data && e.data.type === "viz-height" && e.data.id === "itemgetter_ducktyping") {
    var f = document.getElementById("frame-itemgetter_ducktyping");
    if (f) f.style.height = (e.data.height + 8) + "px";
  }
});
</script>
''')

In [ ]:
# Feature names we'll feed to itemgetter inside collate_dict.
list(tdsf.features)    # ['image', 'label']

**What does the code above do?**

`list(tdsf.features)` gives the feature names as a plain list, `['image', 'label']`. We'll unpack this into `itemgetter` so the getter pulls out exactly the dataset's fields, in order.

In [ ]:
# default_collate stacks a list of per-sample dicts into a dict of batched tensors.
batch = dict(a=[1], b=[2]), dict(a=[3], b=[4])   # two tiny samples
default_collate(batch)    # -> {'a': tensor([1,3]), 'b': tensor([2,4])}

**What does the code above do?**

`default_collate` is PyTorch's built-in batcher. Given a list/tuple of per-sample dicts, it **groups by key and stacks**: the two samples `{'a':[1],'b':[2]}` and `{'a':[3],'b':[4]}` become `{'a': tensor([1,3]), 'b': tensor([2,4])}`. Note the structure stays a **dict** - the values got stacked into tensors, but the result is still keyed by field, not a tuple. To get a tuple, we layer `itemgetter` on top - which is precisely what `collate_dict` does.

### 🎮 Interactive: `default_collate` → the transpose → `collate_dict`'s tuple

Here's the pivot at the heart of batching. A batch arrives as a **list of per-sample dicts**, but the training loop wants a **tuple**. **Step through** (clickable stages + ▶ Play + speed): `default_collate` **groups by key and stacks** (list-of-dicts → dict-of-tensors), then `collate_dict` layers `itemgetter(*ds.features)` on top to read the values out in order — giving you a clean `(xb, yb)` you can unpack directly in `for xb, yb in dl:`.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. Loads ./interactive_viz/default_collate_transpose.html in an
# isolated <iframe> (its CSS/JS can't leak into the notebook). The widget
# broadcasts its own height, so it shows FULL-WIDTH with NO inner scrollbar.
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# To use the published copy instead, swap src for:
#   https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/default_collate_transpose.html
# ============================================================================
from IPython.display import HTML
HTML('''
<iframe id="frame-default_collate_transpose"
        src="interactive_viz/default_collate_transpose.html"
        style="width:100%; min-height:520px; border:1px solid #dde5f2; border-radius:14px;
               box-shadow:0 8px 24px rgba(123,92,214,.12); background:#eef3fb;"
        loading="lazy" title="default_collate transpose and collate_dict"></iframe>
<script>
addEventListener("message", function(e){
  if (e.data && e.data.type === "viz-height" && e.data.id === "default_collate_transpose") {
    var f = document.getElementById("frame-default_collate_transpose");
    if (f) f.style.height = (e.data.height + 8) + "px";
  }
});
</script>
''')

Hugging Face datasets work in dicts, but our training loop wants tuples. The small factory below stitches `default_collate` (does the batching) together with `itemgetter` (does the dict → tuple extraction).

---

# Deep Dive: `collate_dict` - a Closure That Builds a Collate Function

---

## The Problem `collate_dict` Solves

`default_collate` gives us a batched **dict**. The training loop wants a batched **tuple** `(xb, yb)` so it can write `for xb, yb in dl:`. We need a collate function that does both: batch the samples *and* return the result as a tuple in feature order.

But notice a catch: a `DataLoader`'s `collate_fn` is called with just one argument - the list of samples. It has no idea what the feature names are. So we need to *bake in* the feature names ahead of time. That's a job for a **closure / factory function**.

---

## The Complete Code

```python
#|export
def collate_dict(ds):
    get = itemgetter(*ds.features)     # build the getter once, from the dataset's fields
    def _f(b):                          # this is the actual collate_fn the DataLoader calls
        return get(default_collate(b))  # batch -> dict -> tuple
    return _f                           # hand back the configured collate_fn
```

`collate_dict` is a **factory**: it doesn't collate anything itself. You give it a dataset, and it *returns a custom collate function* that already knows that dataset's feature names.

---

## Line-by-Line

### `get = itemgetter(*ds.features)`

`ds.features` is something like `['image', 'label']`. The `*` **unpacks** that list into separate arguments, so this is `itemgetter('image', 'label')`. The result, `get`, is a getter that, applied to a dict, returns `(d['image'], d['label'])` - a tuple, in feature order.

**Why build it here, once?** Because it depends only on the dataset, not on any particular batch. We compute it a single time when the factory runs, and every future batch reuses the same `get`. (Building it fresh inside `_f` on every batch would be wasteful.)

---

### `def _f(b): return get(default_collate(b))`

`_f` is the function the `DataLoader` will actually call, once per batch, with `b` = the list of samples. Read the body inside-out:

1. `default_collate(b)` - batch the samples into a **dict** of stacked tensors, e.g. `{'image': [B,1,28,28], 'label': [B]}`.
2. `get(...)` - apply our pre-built getter to that dict, pulling out the values in feature order as a **tuple**: `([B,1,28,28], [B])`.

So `_f` turns a list of samples straight into `(xb, yb)`.

---

### `return _f`

The factory returns the configured function. This is a **closure**: `_f` "remembers" the `get` from its enclosing `collate_dict` call, even after `collate_dict` has returned. Each call `collate_dict(some_ds)` produces a *new* `_f` bound to *that* dataset's features.

---

## Why a Closure Instead of Just a Function?

We could have written a plain `collate_fn(b)` that hard-codes `itemgetter('image', 'label')`. But then it would only work for datasets with exactly those fields. The closure approach **parameterizes** the collate function by dataset: `collate_dict(any_ds)` adapts to whatever fields `any_ds` has. It's the same pattern as `inplace` (a function that returns a function), used for configuration instead of enhancement.

The interactive below animates the two-step transformation: list-of-dicts → (default_collate) → dict-of-tensors → (itemgetter) → tuple.

### 🎮 Interactive: collate_dict - a list of samples becomes one batched tuple

Step through it: four sample dicts come in; **default_collate** stacks them by key into a dict of batched tensors (the 'transpose'); then **itemgetter** pulls the values out in feature order into the tuple `(xb, yb)` your loop unpacks.

In [ ]:
from IPython.display import IFrame
IFrame("https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/collate_dict_transpose.html", width="100%", height=560)

In [ ]:
#|export
def collate_dict(ds):
    get = itemgetter(*ds.features)        # pre-build getter from feature names
    def _f(b): return get(default_collate(b))   # batch -> dict -> tuple
    return _f

**What does the code above do?**

The exported factory. Call `collate_dict(some_dataset)` and you get back a collate function tailored to that dataset's fields - one that batches with `default_collate` and then extracts a tuple with `itemgetter`. Tagged `#|export`, so it joins the library.

In [ ]:
# Use it: now batches come out as a (xb, yb) TUPLE we can unpack directly.
dlf = DataLoader(tdsf, batch_size=4, collate_fn=collate_dict(tdsf))
xb, yb = next(iter(dlf))    # tuple unpacking works!
xb.shape, yb

**What does the code above do?**

We build a `DataLoader` over the flattened dataset `tdsf`, passing `collate_dict(tdsf)` as the collate function. Now each batch is a **tuple**, so `xb, yb = next(iter(dlf))` unpacks directly - no more `b['image']`/`b['label']`. `xb` holds the (flattened) image batch and `yb` the labels. This tuple form is what every training loop in the later notebooks expects.

---

# Part 6: A Plotting Toolkit for Images

---

The single most valuable debugging habit in deep learning is to **look at your data and your model's outputs**. Matplotlib can do it, but its raw API is verbose and fiddly - especially handling the difference between PIL images, NumPy arrays, and PyTorch tensors (which are channels-*first*, the opposite of what matplotlib wants). So we build a small toolkit, each function layering on the last:

- `show_image(im, ...)` - display **one** image from any source (tensor / PIL / array), handling the axis order automatically
- `subplots(...)` - a friendlier `plt.subplots` with auto-sizing
- `get_grid(n, ...)` - auto-compute a near-square grid for `n` images
- `show_images(ims, ...)` - the one-liner: display a whole list of images with optional titles

In [ ]:
# The most basic way: matplotlib's imshow on a single 2-D array.
b = next(iter(dl))      # dl here yields dict batches; grab one
xb = b['image']         # [16, 1, 28, 28]
img = xb[0]             # [1, 28, 28]  (channels-first)
plt.imshow(img[0]);     # img[0] -> [28,28]; ';' hides the returned object text

**What does the code above do?**

The raw approach, to motivate the toolkit. We grab a batch, take the first image `xb[0]` of shape `[1, 28, 28]`, and call `plt.imshow`. But `imshow` wants a 2-D `[H, W]` array for grayscale, while our tensor is `[C, H, W]` - so we must manually index `img[0]` to drop the channel axis. The trailing `;` suppresses Jupyter printing the returned object.

This is the friction we're about to remove: **PyTorch uses `[C, H, W]` (channels-first); matplotlib expects `[H, W, C]` (channels-last).** Every time you plot a tensor you'd otherwise hand-juggle that. `show_image` does it for you.

---

# Deep Dive: `show_image` - One Function for Every Image Type

---

## The Problem `show_image` Solves

In practice the thing you want to plot might be:

- a **PyTorch tensor** in `[C, H, W]` order, possibly on the GPU, possibly still attached to the autograd graph;
- a **PIL image**;
- a **NumPy array**;
- single-channel (grayscale) or three-channel (RGB).

`plt.imshow` handles none of that gracefully. `show_image` normalizes all of it into the `[H, W]` or `[H, W, C]` NumPy array matplotlib wants, then plots it.

## The Complete Code

```python
#|export
@fc.delegates(plt.Axes.imshow)
def show_image(im, ax=None, figsize=None, title=None, noframe=True, **kwargs):
    if fc.hasattrs(im, ('cpu','permute','detach')):
        im = im.detach().cpu()
        if len(im.shape)==3 and im.shape[0]<5: im = im.permute(1,2,0)
    elif not isinstance(im, np.ndarray): im = np.array(im)
    if im.shape[-1]==1: im = im[...,0]
    if ax is None: _,ax = plt.subplots(figsize=figsize)
    ax.imshow(im, **kwargs)
    if title is not None: ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    if noframe: ax.axis('off')
    return ax
```

---

## Line-by-Line

### `@fc.delegates(plt.Axes.imshow)`

This decorator (deep-dived in the next section) makes `show_image` quietly accept **all** of `imshow`'s keyword arguments - `cmap`, `vmin`, `vmax`, `alpha`, `interpolation`, ... - without us listing them. They flow through `**kwargs` to `ax.imshow`. So `show_image(img, cmap='hot')` just works.

---

### Detecting a tensor by behavior, not type

```python
if fc.hasattrs(im, ('cpu','permute','detach')):
```

**What it does:** `fc.hasattrs` checks whether `im` has *all three* methods `cpu`, `permute`, `detach`. Only PyTorch tensors do. This is **duck typing** again - we ask "can it do tensor things?" rather than `isinstance(im, torch.Tensor)`.

**Why this design:** It's robust to tensor-like objects (subclasses, wrappers) and reads as intent: "if this behaves like a tensor, handle it the tensor way."

---

### Making a tensor matplotlib-safe

```python
        im = im.detach().cpu()
        if len(im.shape)==3 and im.shape[0]<5: im = im.permute(1,2,0)
```

**What it does:** Three fixes in order:
- `.detach()` - drop the autograd history. Matplotlib doesn't want a tensor that's still tracking gradients.
- `.cpu()` - move it off the GPU. Matplotlib can only read CPU memory.
- `.permute(1,2,0)` - reorder `[C, H, W] → [H, W, C]`, but **only** if it's a 3-D tensor whose first axis is small (`< 5`). That `< 5` is a clever heuristic: a real channel count is 1 (grayscale), 3 (RGB), or 4 (RGBA), all `< 5`, whereas a `[H, W]` image or a `28`-row anything wouldn't trip it. It distinguishes "channels-first image" from "already a 2-D image."

**Why this design:** These are the exact three reasons a raw tensor fails to plot. Doing them here, once, means you never think about them again.

---

### Handling PIL / other types

```python
    elif not isinstance(im, np.ndarray): im = np.array(im)
```

**What it does:** If `im` isn't a tensor and isn't already a NumPy array (so: a PIL image, or something array-like), convert it to a NumPy array. `np.array(pil_image)` produces an `[H, W]` or `[H, W, C]` array - already in matplotlib's preferred order.

---

### Squeezing a singleton channel

```python
    if im.shape[-1]==1: im = im[...,0]
```

**What it does:** If the last axis is length 1 (a `[H, W, 1]` grayscale), drop it to get `[H, W]`, which is what `imshow` expects for a single-channel image. `im[...,0]` means "keep all earlier axes, take index 0 of the last."

---

### Axes, drawing, and cleanup

```python
    if ax is None: _,ax = plt.subplots(figsize=figsize)
    ax.imshow(im, **kwargs)
    if title is not None: ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    if noframe: ax.axis('off')
    return ax
```

**What it does:** If no axes was passed, make a fresh figure (so `show_image(img)` works standalone). Draw the image, forwarding any extra kwargs to `imshow`. Optionally set a title. Strip the tick marks and (by default) the whole frame, because axis numbers are meaningless on a picture. Finally **return the axes** - critical, because it lets *callers* place an image into a specific slot of a grid (which `show_images` relies on).

**Why return `ax`?** Returning the axes is what makes the function composable: `show_images` creates a grid of axes and calls `show_image(im, ax=that_slot)` for each one. Without the `ax` parameter and return, you couldn't direct images into a layout.

In [ ]:
#|export
@fc.delegates(plt.Axes.imshow)
def show_image(im, ax=None, figsize=None, title=None, noframe=True, **kwargs):
    "Show a PIL or PyTorch image on `ax`."
    if fc.hasattrs(im, ('cpu','permute','detach')):
        im = im.detach().cpu()                              # off the graph, onto CPU
        if len(im.shape)==3 and im.shape[0]<5:              # looks channels-first?
            im = im.permute(1,2,0)                          # [C,H,W] -> [H,W,C]
    elif not isinstance(im, np.ndarray):
        im = np.array(im)                                   # PIL/other -> numpy
    if im.shape[-1]==1: im = im[...,0]                      # [H,W,1] -> [H,W]
    if ax is None: _,ax = plt.subplots(figsize=figsize)     # make a figure if needed
    ax.imshow(im, **kwargs)
    if title is not None: ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])                    # no tick marks on a picture
    if noframe: ax.axis('off')
    return ax                                               # return ax so callers can compose

**What does the code above do?**

The exported `show_image`, exactly as dissected above: it accepts a tensor / PIL / NumPy image, normalizes it to what matplotlib wants (CPU, channels-last, squeezed grayscale), draws it, tidies the axes, and returns the `ax` so it can be slotted into a grid. The `#|export` ships it to `miniai.datasets`.

In [ ]:
# @fc.delegates means show_image inherited ALL of imshow's params - see the docstring.
help(show_image)

**What does the code above do?**

`help(show_image)` prints the signature and docstring. The striking part: it lists **many parameters we never wrote** - `cmap`, `norm`, `aspect`, `interpolation`, `alpha`, `vmin`, `vmax`, `origin`, ... Those were injected by `@fc.delegates(plt.Axes.imshow)`, which copied `imshow`'s signature onto ours. That's the subject of the next deep-dive.

In [ ]:
# show_image handles the tensor automatically - no manual permute/cpu needed.
show_image(img, figsize=(2,2));

**What does the code above do?**

One call, and the `[1, 28, 28]` tensor `img` is displayed in a compact 2×2-inch figure. Behind the scenes `show_image` moved it to CPU, permuted to `[28, 28, 1]`, squeezed to `[28, 28]`, and plotted it - all the friction from the raw-`imshow` cell, gone.

In [ ]:
# Two images side by side: pass each a specific axes from plt.subplots.
fig, axs = plt.subplots(1, 2)    # 1 row, 2 columns -> axs is an array of 2 axes
show_image(img, axs[0])           # left slot
show_image(xb[1], axs[1]);        # right slot

**What does the code above do?**

To show several images we need several **axes**. `plt.subplots(1, 2)` makes a figure with a 1×2 grid and returns `axs`, an array of two axes. We then call `show_image(im, ax)` once per slot, directing the first image to `axs[0]` and the second to `axs[1]`. This is exactly why `show_image` takes an `ax` argument and returns it - composition into layouts. Manually computing `figsize` for bigger grids gets old fast, which motivates `subplots`.

---

# Deep Dive: `fc.delegates` - Borrowing Another Function's Parameters

---

## The Problem `delegates` Solves

`show_image` ends in `**kwargs` and forwards them to `ax.imshow`. That works at *runtime* - you can pass `cmap='hot'` and it'll reach `imshow`. But it's invisible to *humans and tools*: `help(show_image)` would just show `**kwargs`, with no hint of which options are available. You'd have to go read matplotlib's docs.

`fc.delegates` fixes the **discoverability** gap. It's a fastcore decorator that copies the parameter list (and docs) from the function you delegate to, replacing the bare `**kwargs` in your signature with the real, named parameters of the target.

## What It Does, Concretely

```python
@fc.delegates(plt.Axes.imshow)
def show_image(im, ax=None, ..., **kwargs): ...
```

After this, `show_image`'s signature **advertises** `imshow`'s keyword arguments (`cmap`, `vmin`, `vmax`, `alpha`, ...). They show up in `help()`, in IDE autocomplete, and in nbdev-generated docs. At runtime they still flow through `**kwargs` to `imshow` exactly as before - `delegates` changes the *advertised* signature, not the behavior.

## Why This Matters

It's the difference between a function that's *technically* flexible and one that's *usably* flexible. With `delegates`, a newcomer typing `show_image(img, ` and hitting Tab sees `cmap=`, `alpha=`, and friends offered to them. Without it, those options exist but are hidden behind `**kwargs`. fast.ai uses `delegates` pervasively for exactly this reason - it keeps wrapper functions thin *and* self-documenting.

The optional `keep=True` argument (used in `subplots` below) tells `delegates` to **also keep your own explicit parameters** in the signature, rather than letting the delegated ones replace them. You want that whenever your wrapper adds parameters of its own (like `imsize`) that you don't want to lose.

In [ ]:
#|export
@fc.delegates(plt.subplots, keep=True)
def subplots(
    nrows:int=1,         # rows in the grid
    ncols:int=1,         # cols in the grid
    figsize:tuple=None,  # (w,h) inches; auto-computed if None
    imsize:int=3,        # inches per image, used to auto-size the figure
    suptitle:str=None,   # optional figure-level title
    **kwargs
):
    "A figure and set of subplots to display images of `imsize` inches."
    if figsize is None: figsize = (ncols*imsize, nrows*imsize)   # auto-size
    fig,ax = plt.subplots(nrows, ncols, figsize=figsize, **kwargs)
    if suptitle is not None: fig.suptitle(suptitle)
    if nrows*ncols==1: ax = np.array([ax])   # always return an array, even for 1x1
    return fig,ax

**What does the code above do?**

`subplots` is a friendlier `plt.subplots`. Three conveniences:

1. **Auto-sizing.** If you don't pass `figsize`, it computes one from `imsize`: `(ncols*imsize, nrows*imsize)`. So a 3×3 grid at `imsize=2` becomes a `(6, 6)`-inch figure automatically - you think in "inches per image," not total figure size.
2. **`suptitle`.** A one-arg way to put a title across the whole figure.
3. **Always an array.** Plain `plt.subplots(1,1)` returns a *single* axes object, but `plt.subplots(2,2)` returns an *array* - so code that iterates over axes breaks for the 1×1 case. The line `if nrows*ncols==1: ax = np.array([ax])` wraps the lone axes in an array, so `ax.flat` works uniformly no matter the grid size.

`@fc.delegates(plt.subplots, keep=True)` borrows `plt.subplots`'s own params (`sharex`, `sharey`, ...) into the signature, and `keep=True` keeps our `nrows/ncols/imsize/...` too.

In [ ]:
from nbdev.showdoc import show_doc   # nbdev's formatted documentation renderer

**What does the code above do?**

Imports `show_doc` from nbdev, a helper that renders a function's signature and docstring as a nicely formatted table (used when building documentation sites from notebooks). It's purely for display.

In [ ]:
show_doc(subplots)   # note: delegates copied plt.subplots' params into the signature

**What does the code above do?**

`show_doc(subplots)` displays the formatted documentation. You can see `@fc.delegates(..., keep=True)` at work: the signature shows both *our* parameters (`nrows`, `ncols`, `imsize`, `suptitle`) **and** the delegated ones inherited from `plt.subplots`. Self-documenting, as promised.

In [ ]:
# Use subplots to lay out 8 images in a 3x3 grid (9 slots; last one stays empty).
fig, axs = subplots(3, 3, imsize=1)
imgs = xb[:8]
for ax, img in zip(axs.flat, imgs):   # axs.flat flattens the 2-D axes array to 1-D
    show_image(img, ax)

**What does the code above do?**

`subplots(3, 3, imsize=1)` builds a 3×3 grid of 1-inch slots (auto figsize `(3, 3)`). `axs.flat` flattens the 2-D array of 9 axes into a 1-D sequence so we can iterate it easily. `zip(axs.flat, imgs)` pairs each axes with an image, and we `show_image` each into its slot. We have 8 images but 9 slots, so the last slot is left blank - which motivates `get_grid`, which sizes the grid to the data *and* hides leftover slots.

## `get_grid`: auto-sizing the grid to `n` images

Counting rows and columns by hand is tedious and easy to get wrong (as the empty 9th slot just showed). `get_grid(n)` computes a sensible near-square grid for `n` items and switches off any leftover slots.

### 🎮 Interactive: get_grid - auto-arranging images into a tidy grid

Drag the slider for **n** images and watch `nrows × ncols` recompute (default: a near-square `floor(√n)` rows), with any trailing empty slots switched off. Flip to the *fix rows* / *fix cols* modes to see the other two branches of the logic - including the gotcha where forcing too few rows leaves slots `<` n.

In [ ]:
from IPython.display import IFrame
IFrame("https://shammun.github.io/shammunul-fastai-notes/notebooks/interactive_viz/get_grid_autolayout.html", width="100%", height=540)

In [ ]:
#|export
@fc.delegates(subplots)
def get_grid(
    n:int,              # number of axes needed
    nrows:int=None,     # rows (default: int(sqrt(n)))
    ncols:int=None,     # cols (default: ceil(n/rows))
    title:str=None,     # optional figure title
    weight:str='bold',  # title font weight
    size:int=14,        # title font size
    **kwargs,
):
    "Return a grid of `n` axes, `rows` by `cols`."
    if nrows: ncols = ncols or int(np.floor(n/nrows))   # rows given -> derive cols
    elif ncols: nrows = nrows or int(np.ceil(n/ncols))  # cols given -> derive rows
    else:                                                # neither -> near-square
        nrows = int(math.sqrt(n))
        ncols = int(np.floor(n/nrows))
    fig,axs = subplots(nrows, ncols, **kwargs)
    for i in range(n, nrows*ncols): axs.flat[i].set_axis_off()  # hide extra slots
    if title is not None: fig.suptitle(title, weight=weight, size=size)
    return fig,axs

**What does the code above do?**

`get_grid(n)` picks grid dimensions and returns the figure + axes. The dimension logic has three branches:

| You pass | It computes | Example (n=8) |
|---|---|---|
| `nrows` only | `ncols = floor(n/nrows)` | `nrows=3 → ncols=floor(8/3)=2` |
| `ncols` only | `nrows = ceil(n/ncols)` | `ncols=4 → nrows=ceil(8/4)=2` |
| neither | `nrows = floor(√n)`, `ncols = floor(n/nrows)` | `nrows=2, ncols=4` |

The default (neither given) aims for a **near-square** grid - pleasant to look at. After building the grid, the loop `for i in range(n, nrows*ncols): axs.flat[i].set_axis_off()` **switches off every slot beyond the `n`-th**, so leftover cells don't show as ugly empty axes with ticks.

One honest caveat (the interactive shows it): if you *force* `nrows` such that `nrows × ncols < n` - e.g. `nrows=3` for `n=8` gives only `3×2=6` slots - then two images won't fit. When overriding, make sure the slots cover `n`.

In [ ]:
# Force 3 rows for 8 images: floor(8/3)=2 cols -> only 6 slots (2 imgs won't fit).
fig, axs = get_grid(8, nrows=3, imsize=1)
for ax, img in zip(axs.flat, imgs):
    show_image(img, ax)

**What does the code above do?**

A live demonstration of that very caveat. We ask for `nrows=3` with `n=8`. `get_grid` computes `ncols = floor(8/3) = 2`, giving a `3×2 = 6`-slot grid - **fewer slots than images**. `zip` stops at the shorter sequence (6 slots), so only 6 of the 8 images display. The lesson: when you override grid dimensions, pick values whose product is `≥ n` (here, `ncols=3` would give 9 slots). When in doubt, let `get_grid` choose for you.

In [ ]:
#|export
@fc.delegates(subplots)
def show_images(ims:list,                  # images to show
                nrows:int|None=None,       # rows (auto if None)
                ncols:int|None=None,       # cols (auto if None)
                titles:list|None=None,     # optional per-image titles
                **kwargs):
    "Show all images `ims` as subplots with `rows` using `titles`."
    axs = get_grid(len(ims), nrows, ncols, **kwargs)[1].flat   # make grid, flatten axes
    for im,t,ax in zip_longest(ims, titles or [], axs):        # pair img+title+axes
        show_image(im, ax=ax, title=t)

**What does the code above do?**

`show_images` is the **one-liner** that ties the toolkit together. Given a list of images (and optionally titles), it:

1. `get_grid(len(ims), ...)` - build a grid sized to the number of images; `[1]` takes the axes (it returns `(fig, axs)`), `.flat` flattens them to 1-D.
2. `zip_longest(ims, titles or [], axs)` - pair each image with its title and its axes slot. The key choice here is **`zip_longest`** (not `zip`): if you pass fewer titles than images (or no titles at all - `titles or []` makes it an empty list), `zip_longest` pads the missing titles with `None` instead of stopping early. So `show_image(im, ax=ax, title=None)` simply draws the image with no title. Plain `zip` would have silently dropped images whenever titles ran short.
3. `show_image(im, ax=ax, title=t)` - draw each image into its slot with its (optional) title.

The result: `show_images(my_imgs, titles=my_labels)` displays a labeled grid in one call.

In [ ]:
# Prepare 8 labels to use as titles.
yb = b['label']     # labels from the dict batch
lbls = yb[:8]

**What does the code above do?**

Grab the label tensor from the batch and take the first 8 (`lbls`), to match the 8 images we'll display. These are integer class indices; next we turn them into readable names.

In [ ]:
# Map label indices -> short class names, using itemgetter again.
names = "Top Trouser Pullover Dress Coat Sandal Shirt Sneaker Bag Boot".split()
titles = itemgetter(*lbls)(names)   # (names[9], names[0], ...) in label order
' '.join(titles)

**What does the code above do?**

`names` is a list of 10 short class names (`names[0]='Top'`, ..., `names[9]='Boot'`), built by `.split()`-ing a space-separated string. Then the clever line: `itemgetter(*lbls)(names)`. We *unpack the labels themselves* as the keys - so for `lbls = [9, 0, 0, 3, ...]` this is `itemgetter(9, 0, 0, 3, ...)(names)`, which returns `(names[9], names[0], names[0], names[3], ...) = ('Boot', 'Top', 'Top', 'Dress', ...)`. Here `itemgetter` is indexing a **list** by integer positions, not a dict by string keys - the same tool, duck-typed onto a different container. `' '.join(titles)` prints them as a quick sanity string.

In [ ]:
# The payoff: 8 labeled images in a single line.
show_images(imgs, imsize=1.7, titles=titles)

**What does the code above do?**

The whole toolkit in one call. `show_images(imgs, imsize=1.7, titles=titles)` auto-builds a 2×4 grid (via `get_grid(8)`), sizes it, and draws each Fashion-MNIST image with its class name as a title. *This* is the function you'll reach for constantly: to inspect a batch of training data, to compare predictions against truth, to visualize augmentations. One line, and you can *see* your data.

---

# Part 7: The `DataLoaders` Wrapper

---

Training needs **two** loaders: one for the training split (shuffled) and one for validation (not shuffled). Passing them around as two separate variables is clumsy. `DataLoaders` bundles them into one object with `.train` and `.valid` attributes, plus a factory `from_dd` that builds both straight from a Hugging Face `DatasetDict`.

In [ ]:
#|export
class DataLoaders:
    def __init__(self, *dls): self.train,self.valid = dls[:2]   # first two -> train, valid

    @classmethod
    def from_dd(cls, dd, batch_size, as_tuple=True, **kwargs):
        f = collate_dict(dd['train'])                           # tuple-collate from features
        return cls(*get_dls(*dd.values(), bs=batch_size, collate_fn=f, **kwargs))

**What does the code above do?**

A small but heavily-used class.

**`__init__(self, *dls)`** accepts any number of DataLoaders and stores the first two as `self.train` and `self.valid`. The `*dls` lets you call `DataLoaders(train_dl, valid_dl)` and have them captured as a tuple.

**`from_dd` (a `@classmethod`, i.e. a factory)** builds everything from a `DatasetDict`:
- `f = collate_dict(dd['train'])` - construct the tuple-returning collate function from the training split's features (reusing our deep-dived factory).
- `get_dls(*dd.values(), bs=batch_size, collate_fn=f, **kwargs)` - `dd.values()` is `[train_ds, test_ds]`; `*` unpacks them as the two positional datasets to `get_dls` (our notebook-04 helper that wraps each in a `DataLoader`, shuffling train but not valid). The `*` in front of `get_dls(...)` then unpacks the *returned* pair of loaders straight into `cls(...)`.

So one call - `DataLoaders.from_dd(dsd, batch_size=128)` - turns a raw `DatasetDict` into a tidy object exposing `dls.train` and `dls.valid`, each yielding `(xb, yb)` tuples. This is the object every subsequent notebook's training loop receives.

**Usage:**

```python
dsd = load_dataset('fashion_mnist').with_transform(transformi)
dls = DataLoaders.from_dd(dsd, batch_size=128)
for xb, yb in dls.train: ...   # train
for xb, yb in dls.valid: ...   # validate
```

---

# Summary

---

We built the data layer the rest of the course relies on. The pieces, and why each exists:

**1. Loading (Hugging Face Datasets)**
- `load_dataset_builder` to inspect metadata cheaply; `load_dataset` to download + cache.
- Samples are dicts; **slicing transposes** a batch into a dict of lists.
- `ClassLabel.int2str` maps integer labels back to human-readable names.

**2. Two reusable Python patterns (the high-value ideas)**
- **`@inplace`** - a decorator (a function that returns a function) that appends the easy-to-forget `return b` to in-place transforms.
- **`collate_dict`** - a closure/factory that, given a dataset, returns a collate function which batches with `default_collate` and extracts a feature-ordered **tuple** via `itemgetter`. Powered by **duck typing** (`itemgetter` only needs `[]` access).

**3. Lazy transforms**
- `.with_transform(fn)` applies `fn` **on access**, keeping the source data compact and startup instant.
- `TF.to_tensor` does three jobs: PIL→tensor, scale `0–255 → 0–1`, reorder `HWC→CHW`. `torch.flatten` makes the `[784]` vector a linear model wants.

**4. A plotting toolkit (your debugging eyes)**
- `show_image` - one function, any image type, handles the channels-first/last mismatch (and uses **`fc.delegates`** to inherit all of `imshow`'s options).
- `subplots` / `get_grid` - friendly, auto-sizing layout.
- `show_images` - the one-liner for a labeled grid. Use it constantly.

**5. `DataLoaders`**
- Bundles `train` + `valid`; `from_dd` builds both from a `DatasetDict` in one line.

**The complete pipeline:**
```
load_dataset()  ->  .with_transform(transformi)  ->  DataLoaders.from_dd(...)  ->  for xb,yb in dls.train
```

### Suggested next steps
1. Read the three deep-dives (`@inplace`, `itemgetter`/`collate_dict`, `show_image`/`fc.delegates`) critically - those are the transferable Python ideas, and the most likely place for me to have over- or under-explained. Push back where it doesn't match your understanding.
2. Run `concept-extraction` on this notebook to turn it into durable cards (decorators, closures, duck typing, lazy transforms, channels-first vs -last are all card-worthy atoms).
3. Optionally `/colab` for a GPU-ready version, and `/html` to publish the styled page with the interactives.

---

## Export

The cell below exports every `#|export`-tagged cell to `miniai/datasets.py`, making `inplace`, `collate_dict`, `show_image`, `subplots`, `get_grid`, `show_images`, and `DataLoaders` importable everywhere.

In [ ]:
import nbdev; nbdev.nbdev_export()

**What does the code above do?**

`nbdev.nbdev_export()` scans this notebook for `#|export` cells and writes them into `miniai/datasets.py` (the module named by `#|default_exp datasets` at the top). After running it, any later notebook can `from miniai.datasets import show_images, DataLoaders, ...` and reuse what we built here - which is exactly how the course accumulates a real library, one notebook at a time.